In [3]:
# Classical GAN vs Quantum GAN — 2D toy dataset comparison

This notebook implements and compares a classical GAN (PyTorch) and a quantum (hybrid) GAN using PennyLane + PyTorch on a simple 2D mixture-of-Gaussians toy dataset. It includes training loops, visualization, the MMD metric for quantitative comparison, and runtime measurements. The notebook is designed to run on CPU with modest defaults.
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1) Environment setup & optional installs

Run the cell below only if you need to install dependencies. If running in an environment that already has the required packages (PyTorch, PennyLane, etc.), you can skip this cell. The cell uses a guard to avoid re-installation when packages are present.
</VSCode.Cell>

<VSCode.Cell language="python">
# Optional installation guard — run if you need to install dependencies (uncomment to use)
# Note: for speed and to avoid interruptions, keep installs outside the notebook when possible.

# try:
#     import torch
#     import pennylane as qml
#     print('Dependencies present; skipping install')
# except Exception:
#     # Use pip install; on Windows default shell this cell should work in Jupyter
#     import sys
#     !{sys.executable} -m pip install --upgrade pip
#     !{sys.executable} -m pip install torch torchvision --quiet
#     !{sys.executable} -m pip install pennylane pennylane-qiskit pennylane-qml --quiet
#     !{sys.executable} -m pip install matplotlib seaborn scikit-learn tqdm
#     print('Installed dependencies')

print('Install cell: commented out by default. Enable if needed.')
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2) Imports, seeds & device detection

Import libraries, set reproducible seeds, and detect CPU/GPU and PennyLane availability.
</VSCode.Cell>

<VSCode.Cell language="python">
# Imports and environment detection
import time
import json
import math
import random
from contextlib import contextmanager
from typing import Tuple, List

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

import pennylane as qml
from pennylane import numpy as pnp

from sklearn.metrics import pairwise_distances
from scipy.stats import wasserstein_distance

from tqdm.notebook import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device selection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('PyTorch:', torch.__version__)
print('PennyLane:', qml.__version__)
print('Device:', device)

# Timing helper
@contextmanager
def timer(name: str):
    t0 = time.time()
    yield
    t1 = time.time()
    print(f"Elapsed [{name}]: {t1 - t0:.2f}s")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3) Synthetic 2D dataset (mixture of Gaussians)

We create an N-mode ring (e.g., 8 modes) mixture of Gaussians on a circle. Each mode is a Gaussian centered on a circle point. This dataset is commonly used for toy GAN experiments.
</VSCode.Cell>

<VSCode.Cell language="python">
class RingMixtureDataset(Dataset):
    def __init__(self, n_samples: int = 10000, n_modes: int = 8, radius: float = 2.0, std: float = 0.05):
        self.n_samples = n_samples
        self.n_modes = n_modes
        self.radius = radius
        self.std = std
        self.data = self._generate()
    
    def _generate(self) -> np.ndarray:
        angles = np.random.choice(np.linspace(0, 2 * np.pi, self.n_modes, endpoint=False), size=self.n_samples)
        angles += np.random.normal(scale=self.std, size=self.n_samples)
        x = self.radius * np.cos(angles)
        y = self.radius * np.sin(angles)
        return np.stack([x, y], axis=1).astype(np.float32)
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        return self.data[idx]

# Visualize
dataset = RingMixtureDataset(n_samples=2000, n_modes=8, radius=2.0, std=0.08)
loader = DataLoader(TensorDataset(torch.tensor(dataset.data)), batch_size=128, shuffle=True)

plt.figure(figsize=(5,5))
plt.scatter(dataset.data[:,0], dataset.data[:,1], s=6, alpha=0.6)
plt.title('Real data: 8-mode ring mixture')
plt.axis('equal')
plt.show()
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4) Utility functions: plotting, sampling, MMD, save/load

Helper functions for plotting, sampling, computing MMD, and saving results.
</VSCode.Cell>

<VSCode.Cell language="python">
# Plotting helpers
def plot_scatter(real: np.ndarray, generated: np.ndarray, title: str = '', ax=None, lim=3.0):
    if ax is None:
        plt.figure(figsize=(5,5))
        ax = plt.gca()
    ax.scatter(real[:,0], real[:,1], s=8, alpha=0.4, label='real')
    ax.scatter(generated[:,0], generated[:,1], s=8, alpha=0.6, label='gen')
    ax.set_title(title)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.axis('equal')
    ax.legend()

# Simple MMD (RBF kernel)
def compute_mmd(x: np.ndarray, y: np.ndarray, sigma: float = 1.0) -> float:
    xx = pairwise_distances(x, x, metric='euclidean')
    yy = pairwise_distances(y, y, metric='euclidean')
    xy = pairwise_distances(x, y, metric='euclidean')
    k_xx = np.exp(-xx**2 / (2 * sigma**2))
    k_yy = np.exp(-yy**2 / (2 * sigma**2))
    k_xy = np.exp(-xy**2 / (2 * sigma**2))
    m = x.shape[0]
    n = y.shape[0]
    return k_xx.sum() / (m*m) + k_yy.sum() / (n*n) - 2 * k_xy.sum() / (m*n)

# Save metrics
def save_metrics(path: str, metrics: dict):
    with open(path, 'w') as f:
        json.dump(metrics, f, indent=2)

# Sampling helper for PyTorch generator
def sample_from_generator_torch(gen: nn.Module, n_samples: int, z_dim: int, device='cpu') -> np.ndarray:
    gen.eval()
    with torch.no_grad():
        zs = torch.randn(n_samples, z_dim, device=device)
        out = gen(zs).cpu().numpy()
    gen.train()
    return out
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 5) Classical GAN — PyTorch model definitions

Small MLP generator and discriminator for 2D data.
</VSCode.Cell>

<VSCode.Cell language="python">
# Weight init
def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

# Generator: latent -> 2D
class ClassicalGenerator(nn.Module):
    def __init__(self, z_dim: int = 16, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, 2)
        )
    def forward(self, z):
        return self.net(z)

# Discriminator: 2D -> scalar
class ClassicalDiscriminator(nn.Module):
    def __init__(self, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 6) Classical GAN — training loop

Standard adversarial training using BCE loss and Adam. We keep epochs modest by default.
</VSCode.Cell>

<VSCode.Cell language="python">
def train_classical_gan(real_loader, z_dim=16, epochs=300, lr=2e-4, device='cpu', checkpoint_every=100):
    G = ClassicalGenerator(z_dim=z_dim).to(device)
    D = ClassicalDiscriminator().to(device)
    G.apply(weights_init)
    D.apply(weights_init)

    criterion = nn.BCELoss()
    optG = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    optD = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

    history = {'g_loss': [], 'd_loss': []}
    real_label = 1.
    fake_label = 0.

    for epoch in tqdm(range(epochs), desc='Classical GAN'):
        epoch_g_loss = 0.0
        epoch_d_loss = 0.0
        for batch in real_loader:
            real = batch[0].to(device)
            batch_size = real.size(0)

            # Train D
            D.zero_grad()
            label = torch.full((batch_size,1), real_label, device=device)
            output = D(real)
            lossD_real = criterion(output, label)
            lossD_real.backward()

            # fake
            noise = torch.randn(batch_size, z_dim, device=device)
            fake = G(noise)
            label.fill_(fake_label)
            output = D(fake.detach())
            lossD_fake = criterion(output, label)
            lossD_fake.backward()
            lossD = lossD_real + lossD_fake
            optD.step()

            # Train G
            G.zero_grad()
            label.fill_(real_label)
            output = D(fake)
            lossG = criterion(output, label)
            lossG.backward()
            optG.step()

            epoch_g_loss += lossG.item() * batch_size
            epoch_d_loss += lossD.item() * batch_size

        n = len(real_loader.dataset)
        history['g_loss'].append(epoch_g_loss / n)
        history['d_loss'].append(epoch_d_loss / n)

        # checkpoint / visualize
        if (epoch + 1) % checkpoint_every == 0 or epoch == epochs - 1:
            with torch.no_grad():
                samples = sample_from_generator_torch(G, 1000, z_dim, device=device)
            plt.figure(figsize=(4,4))
            plt.scatter(dataset.data[:,0][:1000], dataset.data[:,1][:1000], s=6, alpha=0.4, label='real')
            plt.scatter(samples[:,0], samples[:,1], s=6, alpha=0.6, label='gen')
            plt.title(f'Classical GAN samples — epoch {epoch+1}')
            plt.axis('equal')
            plt.show()

    return G, D, history

# Example run with modest epochs
z_dim = 16
real_loader = DataLoader(TensorDataset(torch.tensor(dataset.data)), batch_size=128, shuffle=True)

with timer('Classical GAN training'):
    G_classical, D_classical, hist_classical = train_classical_gan(real_loader, z_dim=z_dim, epochs=200, lr=2e-4, device=device, checkpoint_every=100)
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 7) Classical GAN — evaluation (MMD, plots)

Compute MMD and visualize results for the classical generator.
</VSCode.Cell>

<VSCode.Cell language="python">
# Sample many points from classical generator
samples_classical = sample_from_generator_torch(G_classical, 5000, z_dim, device=device)

mmd_classical = compute_mmd(dataset.data[np.random.choice(len(dataset), 5000, replace=False)], samples_classical, sigma=0.5)
print('Classical GAN MMD:', mmd_classical)

plt.figure(figsize=(10,4))
ax1 = plt.subplot(1,2,1)
plt.scatter(dataset.data[:,0][:2000], dataset.data[:,1][:2000], s=6, alpha=0.4)
plt.title('Real data')
plt.axis('equal')
ax2 = plt.subplot(1,2,2)
plt.scatter(samples_classical[:,0], samples_classical[:,1], s=6, alpha=0.6, color='orange')
plt.title('Classical GAN generated')
plt.axis('equal')
plt.show()
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 8) Quantum GAN — PennyLane device & variational generator

We use PennyLane to build a small variational quantum circuit that maps a classical latent vector z to two expectation values (mapped to 2D coordinates). We will wrap the qnode using pennylane.qnn.TorchLayer so it can be trained with PyTorch. If a GPU is not available or PennyLane isn't installed, this cell will still run using the default.qubit simulator.
</VSCode.Cell>

<VSCode.Cell language="python">
# Quantum generator settings
n_qubits = 2
q_depth = 3
q_z_dim = 4  # latent dimension for quantum generator (will be embedded)

# PennyLane device
dev = qml.device('default.qubit', wires=n_qubits)

# Variational circuit: embed z via rotations, then variational layers
n_params = q_depth * n_qubits * 3  # Rx,Ry,Rz per qubit per layer (example)

@qml.qnode(dev, interface='torch')
def qnode(inputs, weights):
    # inputs: tensor of size q_z_dim
    # weights: flat vector of parameters shaped (q_depth, n_qubits, 3)
    # embed inputs into rotations on first qubit(s)
    for i in range(n_qubits):
        qml.RX(inputs[i % inputs.shape[0]], wires=i)
    # variational layers
    weights = weights.reshape(q_depth, n_qubits, 3)
    for l in range(q_depth):
        for q in range(n_qubits):
            qml.RX(weights[l,q,0], wires=q)
            qml.RY(weights[l,q,1], wires=q)
            qml.RZ(weights[l,q,2], wires=q)
        # entangling
        for q in range(n_qubits-1):
            qml.CNOT(wires=[q, q+1])
    # return expectations mapped to 2D
    return [qml.expval(qml.PauliZ(0)), qml.expval(qml.PauliZ(1))]

# Create TorchLayer
weight_shapes = {'weights': (n_params,)}
qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)

# Small classical head to scale outputs
class QuantumGeneratorTorch(nn.Module):
    def __init__(self, qlayer, z_dim=4, hidden=16):
        super().__init__()
        self.qlayer = qlayer
        self.head = nn.Sequential(
            nn.Linear(2, hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, 2)
        )
    def forward(self, z):
        # z: (batch, z_dim)
        # reduce to size 2 by simple linear projection for embedding
        # we will use first 2 elements for rotation angles
        angles = z[:, :2]
        out_q = torch.stack([self.qlayer(a) for a in angles])
        return self.head(out_q)

# Instantiate quantum generator and discriminator
G_quantum = QuantumGeneratorTorch(qlayer, z_dim=q_z_dim).to(device)
D_quantum = ClassicalDiscriminator().to(device)

# Initialize parameters (PennyLane params are inside the TorchLayer)
print('Quantum generator and discriminator created')
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 9) Quantum GAN — training loop (hybrid)

We train the discriminator using standard BCE and the generator by backpropagating through the PennyLane TorchLayer. This is a hybrid training loop aligning with the classical GAN procedure for fair comparison.
</VSCode.Cell>

<VSCode.Cell language="python">
def train_quantum_gan(real_loader, G, D, z_dim=4, epochs=300, lr=2e-4, device='cpu', checkpoint_every=100):
    G = G.to(device)
    D = D.to(device)
    criterion = nn.BCELoss()
    optG = optim.Adam(G.parameters(), lr=lr)
    optD = optim.Adam(D.parameters(), lr=lr)

    history = {'g_loss': [], 'd_loss': []}
    real_label = 1.
    fake_label = 0.

    for epoch in tqdm(range(epochs), desc='Quantum GAN'):
        epoch_g_loss = 0.0
        epoch_d_loss = 0.0
        for batch in real_loader:
            real = batch[0].to(device)
            batch_size = real.size(0)

            # Train D
            D.zero_grad()
            label = torch.full((batch_size,1), real_label, device=device)
            output = D(real)
            lossD_real = criterion(output, label)
            lossD_real.backward()

            # fake
            noise = torch.randn(batch_size, z_dim, device=device)
            fake = G(noise)
            label.fill_(fake_label)
            output = D(fake.detach())
            lossD_fake = criterion(output, label)
            lossD_fake.backward()
            lossD = lossD_real + lossD_fake
            optD.step()

            # Train G
            G.zero_grad()
            label.fill_(real_label)
            output = D(fake)
            lossG = criterion(output, label)
            lossG.backward()
            optG.step()

            epoch_g_loss += lossG.item() * batch_size
            epoch_d_loss += lossD.item() * batch_size

        n = len(real_loader.dataset)
        history['g_loss'].append(epoch_g_loss / n)
        history['d_loss'].append(epoch_d_loss / n)

        if (epoch + 1) % checkpoint_every == 0 or epoch == epochs - 1:
            with torch.no_grad():
                zs = torch.randn(1000, z_dim, device=device)
                samples = G(zs).cpu().numpy()
            plt.figure(figsize=(4,4))
            plt.scatter(dataset.data[:,0][:1000], dataset.data[:,1][:1000], s=6, alpha=0.4, label='real')
            plt.scatter(samples[:,0], samples[:,1], s=6, alpha=0.6, label='gen')
            plt.title(f'Quantum GAN samples — epoch {epoch+1}')
            plt.axis('equal')
            plt.show()

    return G, D, history

# Example run with modest epochs
with timer('Quantum GAN training'):
    G_quantum_trained, D_quantum_trained, hist_quantum = train_quantum_gan(real_loader, G_quantum, D_quantum, z_dim=q_z_dim, epochs=200, lr=2e-4, device=device, checkpoint_every=100)
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 10) Quantum GAN — evaluation (MMD, plots)

Sample from the trained quantum generator, compute MMD, and visualize.
</VSCode.Cell>

<VSCode.Cell language="python">
# Sample from quantum generator
G_quantum_trained.eval()
with torch.no_grad():
    zs = torch.randn(5000, q_z_dim, device=device)
    samples_quantum = G_quantum_trained(zs).cpu().numpy()

mmd_quantum = compute_mmd(dataset.data[np.random.choice(len(dataset), 5000, replace=False)], samples_quantum, sigma=0.5)
print('Quantum GAN MMD:', mmd_quantum)

plt.figure(figsize=(10,4))
ax1 = plt.subplot(1,2,1)
plt.scatter(dataset.data[:,0][:2000], dataset.data[:,1][:2000], s=6, alpha=0.4)
plt.title('Real data')
plt.axis('equal')
ax2 = plt.subplot(1,2,2)
plt.scatter(samples_quantum[:,0], samples_quantum[:,1], s=6, alpha=0.6, color='green')
plt.title('Quantum GAN generated')
plt.axis('equal')
plt.show()
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 11) Comparison: MMD, loss curves, runtime

Compare MMD values, plot loss curves for both runs, and show side-by-side sample plots.
</VSCode.Cell>

<VSCode.Cell language="python">
# Collect metrics
metrics = {
    'mmd_classical': float(mmd_classical),
    'mmd_quantum': float(mmd_quantum),
    'epochs_classical': len(hist_classical['g_loss']),
    'epochs_quantum': len(hist_quantum['g_loss']),
}

print('Metrics:', metrics)

# Loss curves
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(hist_classical['g_loss'], label='G_classical')
plt.plot(hist_classical['d_loss'], label='D_classical')
plt.title('Classical GAN losses')
plt.legend()

plt.subplot(1,2,2)
plt.plot(hist_quantum['g_loss'], label='G_quantum')
plt.plot(hist_quantum['d_loss'], label='D_quantum')
plt.title('Quantum GAN losses')
plt.legend()
plt.show()

# Side-by-side scatter
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.scatter(dataset.data[:,0][:2000], dataset.data[:,1][:2000], s=6, alpha=0.4)
plt.title('Real')
plt.axis('equal')
plt.subplot(1,3,2)
plt.scatter(samples_classical[:,0], samples_classical[:,1], s=6, alpha=0.6, color='orange')
plt.title('Classical')
plt.axis('equal')
plt.subplot(1,3,3)
plt.scatter(samples_quantum[:,0], samples_quantum[:,1], s=6, alpha=0.6, color='green')
plt.title('Quantum')
plt.axis('equal')
plt.show()

# Save metrics
save_metrics('gan_comparison_metrics.json', metrics)
print('Saved metrics to gan_comparison_metrics.json')
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 12) Notes and next steps

- This notebook uses PennyLane's TorchLayer for a simple hybrid quantum generator. On larger problems or real hardware, more careful circuit design and gradient methods (parameter-shift rules) are required.
- For CPU-only runs, keep epochs small. For strict comparisons, tune hyperparameters equally for both models.
- Extensions: use different evaluation metrics (FID for images), experiment with quantum encodings, or run generator optimization with gradient-free methods if using a black-box quantum backend.


SyntaxError: invalid decimal literal (2419734490.py, line 3)